<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/prompt_generation_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Meta-Prompt를 이용한 Prompt Generation 예제

## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )

# Reference : https://platform.openai.com/docs/guides/prompt-generation

In [1]:
!pip install openai

In [2]:
!pip show openai

Name: openai
Version: 2.14.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: 


## OpenAI API Key 설정(2026 기준으로 코드를 변경하였다)

In [4]:

from openai import OpenAI

# 1. 클라이언트 초기화 (가장 권장되는 방식)
client = OpenAI(
    api_key="Input Your API KEY"
)


# 1. Meta-prompts

In [5]:
# META_PROMPT = """
# Given a current prompt and a change description, produce a detailed system prompt to guide a language model in completing the task effectively.

# Your final output will be the full corrected prompt verbatim. However, before that, at the very beginning of your response, use <reasoning> tags to analyze the prompt and determine the following, explicitly:
# <reasoning>
# - Simple Change: (yes/no) Is the change description explicit and simple? (If so, skip the rest of these questions.)
# - Reasoning: (yes/no) Does the current prompt use reasoning, analysis, or chain of thought?
#     - Identify: (max 10 words) if so, which section(s) utilize reasoning?
#     - Conclusion: (yes/no) is the chain of thought used to determine a conclusion?
#     - Ordering: (before/after) is the chain of though located before or after
# - Structure: (yes/no) does the input prompt have a well defined structure
# - Examples: (yes/no) does the input prompt have few-shot examples
#     - Representative: (1-5) if present, how representative are the examples?
# - Complexity: (1-5) how complex is the input prompt?
#     - Task: (1-5) how complex is the implied task?
#     - Necessity: ()
# - Specificity: (1-5) how detailed and specific is the prompt? (not to be confused with length)
# - Prioritization: (list) what 1-3 categories are the MOST important to address.
# - Conclusion: (max 30 words) given the previous assessment, give a very concise, imperative description of what should be changed and how. this does not have to adhere strictly to only the categories listed
# </reasoning>

# # Guidelines

# - Understand the Task: Grasp the main objective, goals, requirements, constraints, and expected output.
# - Minimal Changes: If an existing prompt is provided, improve it only if it's simple. For complex prompts, enhance clarity and add missing elements without altering the original structure.
# - Reasoning Before Conclusions**: Encourage reasoning steps before any conclusions are reached. ATTENTION! If the user provides examples where the reasoning happens afterward, REVERSE the order! NEVER START EXAMPLES WITH CONCLUSIONS!
#     - Reasoning Order: Call out reasoning portions of the prompt and conclusion parts (specific fields by name). For each, determine the ORDER in which this is done, and whether it needs to be reversed.
#     - Conclusion, classifications, or results should ALWAYS appear last.
# - Examples: Include high-quality examples if helpful, using placeholders [in brackets] for complex elements.
#    - What kinds of examples may need to be included, how many, and whether they are complex enough to benefit from placeholders.
# - Clarity and Conciseness: Use clear, specific language. Avoid unnecessary instructions or bland statements.
# - Formatting: Use markdown features for readability. DO NOT USE ``` CODE BLOCKS UNLESS SPECIFICALLY REQUESTED.
# - Preserve User Content: If the input task or prompt includes extensive guidelines or examples, preserve them entirely, or as closely as possible. If they are vague, consider breaking down into sub-steps. Keep any details, guidelines, examples, variables, or placeholders provided by the user.
# - Constants: DO include constants in the prompt, as they are not susceptible to prompt injection. Such as guides, rubrics, and examples.
# - Output Format: Explicitly the most appropriate output format, in detail. This should include length and syntax (e.g. short sentence, paragraph, JSON, etc.)
#     - For tasks outputting well-defined or structured data (classification, JSON, etc.) bias toward outputting a JSON.
#     - JSON should never be wrapped in code blocks (```) unless explicitly requested.

# The final prompt you output should adhere to the following structure below. Do not include any additional commentary, only output the completed system prompt. SPECIFICALLY, do not include any additional messages at the start or end of the prompt. (e.g. no "---")

# [Concise instruction describing the task - this should be the first line in the prompt, no section header]

# [Additional details as needed.]

# [Optional sections with headings or bullet points for detailed steps.]

# # Steps [optional]

# [optional: a detailed breakdown of the steps necessary to accomplish the task]

# # Output Format

# [Specifically call out how the output should be formatted, be it response length, structure e.g. JSON, markdown, etc]

# # Examples [optional]

# [Optional: 1-3 well-defined examples with placeholders if necessary. Clearly mark where examples start and end, and what the input and output are. User placeholders as necessary.]
# [If the examples are shorter than what a realistic example is expected to be, make a reference with () explaining how real examples should be longer / shorter / different. AND USE PLACEHOLDERS! ]

# # Notes [optional]

# [optional: edge cases, details, and an area to call or repeat out specific important considerations]
# [NOTE: you must start with a <reasoning> section. the immediate next token you produce should be <reasoning>]
# """.strip()

META_PROMPT = """
작업 설명이나 기존 프롬프트를 바탕으로, 언어 모델이 해당 작업을 효과적으로 수행할 수 있도록 자세한 시스템 프롬프트를 생성하세요.

최종 출력은 수정된 전체 프롬프트를 그대로 출력해야 합니다. 다만, 그 전에 응답의 가장 앞부분에  <reasoning> 태그를 사용하여 프롬프트를 분석하고 다음 항목들을 명시적으로 판단하세요:
<reasoning>
- 간단한 수정: (예/아니오) 변경 요청이 명확하고 단순한가요? (그럴 경우 아래 항목은 생략 가능)
- 추론 포함 여부: (예/아니오) 현재 프롬프트에 추론, 분석 또는 사고의 흐름이 포함되어 있나요?
    - 식별: (최대 10단어) 그렇다면 어느 부분에서 추론이 사용되나요?
    - 결론 포함: (예/아니오) 사고의 흐름이 결론을 도출하는 데 사용되나요?
    - 순서: (이전/이후) 사고의 흐름이 결론보다 앞에 위치하나요, 뒤에 위치하나요?
- 구조: (예/아니오) 입력된 프롬프트에 명확한 구조가 있나요?
- 예시 포함: (예/아니오) 프롬프트에 몇 개의 예시가 포함되어 있나요?
    - 대표성: (1-5) 포함된 경우, 예시가 얼마나 대표적인가요?
- 복잡성: (1-5) 입력된 프롬프트가 얼마나 복잡한가요?
    - 작업 난이도: (1-5) 요구된 작업이 얼마나 복잡한가요?
    - 필요성: ()
- 구체성: (1-5) 프롬프트가 얼마나 구체적이고 명확한가요? (길이와는 무관)
- 우선순위: (리스트) 가장 중요하게 다뤄야 할 1~3개의 항목은 무엇인가요?
- 결론: (최대 30단어) 위 분석을 바탕으로, 어떤 점을 어떻게 변경해야 하는지 간결하게 지시하세요. 이 항목은 반드시 위의 카테고리만 고수하지 않아도 됩니다.
 </reasoning>

# 가이드라인

- 작업 이해하기: 작업의 주요 목표, 목적, 요구사항, 제약 조건, 기대되는 출력을 정확히 파악하세요.
- 최소한의 수정: 기존 프롬프트가 제공된 경우, 단순한 경우에만 개선하세요. 복잡한 프롬프트는 원래 구조를 변경하지 않고 명확성을 높이고 누락된 요소를 추가하세요.
- 결론보다 추론이 먼저: 결론에 도달하기 전에 반드시 추론 과정을 유도하세요. 사용자가 예시에서 추론이 결론 뒤에 나오도록 제시한 경우, 그 순서를 반드시 반대로 바꾸세요. 예시는 절대로 결론부터 시작하지 마세요.
    - 추론 순서: 프롬프트 내에서 추론 부분과 결론 부분을 구분하세요. 각 부분의 순서를 파악하고, 필요하다면 순서를 반전시키세요.
    - 결론이나 결과는 항상 마지막에 와야 합니다.
- 예시: 도움이 된다면 고품질 예시를 포함하세요. 복잡한 요소는 [대괄호] 안에 플레이스홀더로 표시하세요.
    - 어떤 종류의 예시가 필요한지, 몇 개가 적절한지, 복잡한 요소는 추상화가 필요한지 고려하세요.
- 명확성과 간결성: 불필요한 설명이나 형식적인 문장은 피하고, 명확하고 구체적인 언어를 사용하세요.
- 포맷팅: 가독성을 위해 마크다운을 사용하되, 코드 블록( ```)은 사용자가 명시적으로 요청한 경우에만 사용하세요.
- 사용자 내용 보존: 입력된 작업 설명이나 프롬프트에 지침이나 예시가 포함되어 있다면, 가능한 한 그대로 유지하거나 충실히 반영하세요. 만약 설명이 불분명하다면, 작업을 하위 단계로 나누어 명확하게 구성하세요. 변수, 예시, 지침, 플레이스홀더 등도 포함된 그대로 유지하세요.
- 상수는 포함: 평가 기준, 예시, 지침 등 상수로 간주되는 내용은 프롬프트에 포함하세요. 이들은 프롬프트 인젝션에 취약하지 않으므로 안전합니다.
- 출력 형식: 가장 적절한 출력 형식을 명확히 지정하세요. 예: 짧은 문장, 단락, JSON 등.
    - 구조화된 데이터를 출력하는 작업의 경우 JSON 형식을 우선 고려하세요.
    - 단, JSON은 코드 블록으로 감싸지 마세요. 사용자가 요청한 경우에만 예외로 허용하세요.

최종 출력 프롬프트는 아래 구조를 반드시 따라야 합니다. 추가적인 설명은 포함하지 마세요. 시스템 프롬프트만 출력하세요. 특히 프롬프트 시작이나 끝에 어떠한 메시지도 추가하지 마세요. (예: "---" 금지)

[작업을 설명하는 간결한 지시문 - 프롬프트의 첫 줄에 위치해야 하며, 섹션 제목은 사용하지 마세요]

[필요한 추가 세부사항]

[선택적으로 자세한 단계 설명을 위한 제목이나 불릿 포인트 섹션 포함 가능]

# Steps [선택사항]

[선택사항: 작업을 수행하는 데 필요한 단계들을 자세히 설명한 내용]

# 출력 형식

[출력이 어떤 형식으로 되어야 하는지 구체적으로 명시하세요. 예: 응답 길이, 구조 (JSON, 마크다운 등)]

# 예시 [선택사항]

[선택사항: 1~3개의 잘 정의된 예시를 포함하세요. 필요시 플레이스홀더를 사용하세요. 예시의 시작과 끝, 입력과 출력을 명확히 구분하세요. 필요한 경우 플레이스홀더를 사용하세요.]
[예시가 실제 예시에 비해 짧은 경우, 괄호 안에 실제 예시는 더 길거나 짧아야 한다는 설명을 추가하세요. 반드시 플레이스홀더를 사용하세요!]

# 참고사항 [선택사항]

[선택사항: 엣지 케이스나 세부사항, 또는 중요한 고려 사항을 강조하거나 반복해서 언급하는 공간입니다.]
[주의: 반드시 <reasoning> 섹션으로 시작해야 합니다. 다음에 나올 첫 번째 토큰은 <reasoning>이어야 합니다.]
""".strip()


In [8]:
def generate_prompt(task_or_prompt: str):
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": META_PROMPT,
            },
            {
                "role": "user",
                #"content": "Task, Goal, or Current Prompt:\n" + task_or_prompt,
                "content": "작업, 목표 또는 현재 프롬프트::\n" + task_or_prompt,
            },
        ],
    )

    return completion.choices[0].message.content

In [7]:
code_prompt = generate_prompt(
    "사용자 요청을 토대로 프로그래밍 코드 생성"
)
print(code_prompt)

<reasoning>
- 간단한 수정: 아니오
- 추론 포함 여부: 아니오
- 구조: 아니오
- 예시 포함: 아니오
- 복잡성: (1-5) 2
    - 작업 난이도: (1-5) 2
- 구체성: (1-5) 2
- 우선순위: 
    1. 사용자 요청의 구체적인 세부정보 명시
    2. 출력 형식 명시
    3. 예시 포함
- 결론: 사용자 요청과 출력 형식을 구체적으로 정의하고 예시를 포함해야 합니다.
</reasoning>

사용자의 요청에 따라 프로그래밍 코드를 생성하세요.

요청 사항을 명확히 지정하고, 코드 작성을 위한 출력 형식과 언어를 포함하십시오.

# Steps
- 사용자로부터 어떤 코드가 필요한지 명확히 질문하십시오.
- 사용자 요청에 따라 적절한 프로그래밍 언어를 선택하십시오.
- 요청된 기능을 충분히 수행하는 코드를 작성하십시오.

# 출력 형식
- 생성된 코드는 [프로그래밍 언어]로 작성되어야 하며, 주석을 포함하여 이해하기 쉽게 해야 합니다.

# 예시
**입력:** "파이썬으로 두 수의 합을 계산하는 함수를 만들어 주세요."  
**출력:** 
```python
def add_numbers(a, b):
    """두 수를 더하는 함수"""
    return a + b
```
(지금 제공된 예시는 실제 예시에 비해 간단할 수 있습니다. 복잡한 요청일 경우 더 긴 예시가 필요합니다.)


In [9]:
code_generation_system_prompt = """
사용자 요청에 따라 프로그래밍 코드를 생성하세요. 사용된 프로그래밍 언어나 필요 조건이 있다면 반드시 준수하세요. 주석, 변수명, 함수명 등은 요청의 맥락상 적절하게 선택하세요.

요청 내용이 불분명할 경우, 짧은 코드와 함께 간략한 설명을 추가하고 가정한 내용을 명확히 표시하세요.

요청에 특별한 언어 또는 형식이 명확하지 않다면 Python을 우선 사용하세요.

# 출력 형식

가능하면 간결한 코드(짧은 함수 또는 스크립트)로 작성하세요. 코드 블록(```) 안에 코드를 포함하고, 코드 위에 언어명을 명시하세요(예: Python).

# 예시

입력: 숫자 리스트의 합계를 구하는 코드를 작성해줘.
출력:
Python
```
def sum_numbers(numbers):
    return sum(numbers)
```

입력: 주어진 문자열을 뒤집는 자바스크립트 함수를 만들어 주세요.
출력:
JavaScript
```
function reverseString(str) {
    return str.split('').reverse().join('');
}
```
"""

In [10]:
user_request = "1부터 100까지 중에서 소수가 몇개인지를 계산하는 python 코드를 작성해줘,"
messages = [
    {"role": "system", "content": code_generation_system_prompt},
    {"role": "user", "content": user_request},
]
response = client.chat.completions.create(
    model='gpt-4o-mini', messages=messages, temperature=0.0
)
print(response.choices[0].message.content)

Python
```python
def count_primes(n):
    primes = []
    for num in range(2, n + 1):
        is_prime = True
        for i in range(2, int(num**0.5) + 1):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(num)
    return len(primes)

prime_count = count_primes(100)
print(prime_count)
``` 

이 코드는 1부터 100까지의 소수를 계산하여 그 개수를 반환합니다. `count_primes` 함수는 주어진 숫자 `n`까지의 소수를 찾고, 그 개수를 세어 반환합니다.


In [11]:
stock_analyze_prompt = generate_prompt(
    "삼성전자의 미래 주가를 예측하기 위한 보고서 작성"
)
print(stock_analyze_prompt)

<reasoning>
- 간단한 수정: 아니오 변경 요청이 명확하고 단순한가요? 
- 추론 포함 여부: 예 현재 프롬프트에 추론, 분석 또는 사고의 흐름이 포함되어 있나요?
    - 식별: 미래 주가 예측을 위한 분석
    - 결론 포함: 아니오 사고의 흐름이 결론을 도출하는 데 사용되나요?
    - 순서: 이전
- 구조: 아니오 입력된 프롬프트에 명확한 구조가 있나요?
- 예시 포함: 아니오 프롬프트에 몇 개의 예시가 포함되어 있나요?
    - 대표성: 1
- 복잡성: 3 입력된 프롬프트가 얼마나 복잡한가요?
    - 작업 난이도: 4 요구된 작업이 얼마나 복잡한가요?
    - 필요성: (없음)
- 구체성: 2 프롬프트가 얼마나 구체적이고 명확한가요?
- 우선순위: 
    1. 데이터 분석 방법론
    2. 예측 시나리오
    3. 보고서 형식과 구조
- 결론: 보고서의 형식, 데이터 분석 방법, 예측 시나리오를 명확히 하고 구조화하여 제시하세요.
</reasoning>

삼성전자의 미래 주가를 예측하기 위한 보고서를 작성하세요.

- 보고서에는 아래 항목들이 포함되어야 합니다:
    1. 서론: 삼성전자의 현재 시장 상황과 경제적 배경.
    2. 데이터 분석: 주가 예측에 사용될 데이터 출처와 분석 방법.
    3. 예측 시나리오: 다양한 변수에 따른 주가 예측 시나리오 제시.
    4. 결론: 조망되는 주가 변화와 투자 전략 제안.

# 출력 형식

보고서는 마크다운 형식으로 구성되어야 하며, 각 섹션은 제목으로 구분되고 내용이 포함되어야 합니다.


In [14]:
stock_analyze_prompt_system_prompt = """
삼성전자의 미래 주가를 예측하기 위한 구조화된 보고서를 작성하세요. 다음의 단계별 분석과 종합적 결론을 반드시 포함해야 합니다. 모든 주요 추론 과정(시장 환경, 기업 내부 역량, 재무 데이터, 업계 트렌드 등)에 논리적 근거와 데이터 기반의 설명을 제공하고, 결론(미래 주가 예측)은 분석 및 추론 다음에 명확히 제시하십시오.

- 최신 글로벌 및 국내 시장 동향, 경제 환경 분석
- 삼성전자의 산업 내 위치, 주요 경쟁자와 비교
- 삼성전자 주요 사업부(반도체, 스마트폰, 가전 등)의 사업 환경, 성장성, 위험 요인
- 과거 주가 변동과 주요 이슈 분석
- 최근 재무 데이터(매출, 영업이익, 부채, 투자 등) 평가
- 투자자 및 애널리스트의 시각 및 기대치 요약
- 미래 성장동력(기술, M&A, 글로벌 사업 등) 전망과 잠재 리스크 분석
- 앞선 모든 분석을 종합한 미래 주가 추정 및 그 근거, 적용한 예측 모델(있다면) 상세 설명

# 출력 형식

- 보고서는 구조화된 마크다운 양식으로 작성
- 각 대주제별 소제목(h2, h3 등)과 명확한 구분 사용
- 논리적 근거 및 데이터는 각각 인용이나 참고 형태로 명시
- 결론(미래 주가 예측)은 반드시 마지막에 "미래 주가 전망 및 결론" 섹션에서 제시

# 예시

## 1. 글로벌 및 국내 시장 분석
- [여기에 최근 글로벌 경기 동향, 반도체/IT산업 트렌드, 경제성장률 전망 등 서술. 예시: "2023년 글로벌 반도체 시장은 수요 회복과 공급망 불확실성 완화로 2.5% 성장할 것으로 전망된다."]

## 2. 삼성전자의 산업 내 위치 및 경쟁 분석
- [삼성전자와 TSMC, 애플 등 경쟁사 비교. "삼성전자는 메모리 분야 점유율 44%로 업계 1위, 파운드리 부문에서는 TSMC의 시장 지배력에 미치지 못함."]

...
(각 단계별로 h2, h3 제목과 함께 근거 및 데이터, 분석 내용을 상세히 작성하세요.)

## 미래 주가 전망 및 결론
- [앞선 분석을 종합해 2024년 연말 삼성전자 목표 주가는 75,000원(플레이스홀더)으로 예측된다. 다양한 변동성, 거시환경 리스크, 사업부문 별 성장 기대감 등을 고려했음.]

# 참고사항

- 구체적 수치, 인용 데이터, 모델 명칭은 “플레이스홀더”로 표기 가능.
- 추론 과정이 누락되지 않도록 단계별로 논리적 연결고리 명확히 서술.
- 결론은 반드시 모든 추론 및 분석 이후에만 제시.
- 실제 투자 조언이 아님을 각주 등으로 밝히세요.
"""

In [15]:
user_request = "삼성전자의 미래 주가 예측을 위한 보고서를 작성해줘."
messages = [
    {"role": "system", "content": stock_analyze_prompt_system_prompt},
    {"role": "user", "content": user_request},
]
response = client.chat.completions.create(
    model='gpt-4o-mini', messages=messages, temperature=0.0
)
print(response.choices[0].message.content)

# 삼성전자의 미래 주가 예측 보고서

## 1. 글로벌 및 국내 시장 분석
- 2023년 글로벌 경제는 인플레이션 압력 완화와 중앙은행의 금리 인하 기대감으로 인해 점진적인 회복세를 보일 것으로 전망된다. 특히, 반도체 및 IT 산업은 수요 회복과 공급망 불확실성 완화로 인해 2.5% 성장할 것으로 예상된다. 
- 국내 시장에서는 반도체 산업이 여전히 주요 성장 동력으로 작용하고 있으며, 2023년 한국의 GDP 성장률은 약 2.0%로 예상된다. 이는 삼성전자의 주요 사업 부문에 긍정적인 영향을 미칠 것으로 보인다.

## 2. 삼성전자의 산업 내 위치 및 경쟁 분석
- 삼성전자는 메모리 반도체 분야에서 44%의 시장 점유율을 기록하며 업계 1위를 차지하고 있다. 그러나 파운드리 부문에서는 TSMC의 시장 지배력에 미치지 못하고 있으며, 이 부문에서의 경쟁력 강화를 위한 투자가 필요하다.
- 주요 경쟁사인 애플은 자사 칩 설계 및 생산을 통해 독립성을 강화하고 있으며, 이는 삼성전자의 스마트폰 부문에 위협이 될 수 있다. 또한, 인텔과 AMD는 고성능 반도체 시장에서의 경쟁을 강화하고 있다.

## 3. 삼성전자 주요 사업부 분석
### 3.1 반도체
- 반도체 부문은 삼성전자의 매출의 약 50%를 차지하며, AI 및 데이터 센터 수요 증가로 인해 성장 가능성이 높다. 그러나 공급 과잉 및 가격 하락 위험이 존재한다.
### 3.2 스마트폰
- 스마트폰 부문은 프리미엄 시장에서의 경쟁이 치열해지고 있으며, 특히 애플과의 경쟁이 심화되고 있다. 그러나 중저가 시장에서의 성장은 긍정적이다.
### 3.3 가전
- 가전 부문은 지속 가능한 제품에 대한 수요 증가로 인해 성장 가능성이 있으며, 스마트 홈 기술의 발전이 기여할 것으로 보인다.

## 4. 과거 주가 변동 및 주요 이슈 분석
- 삼성전자의 주가는 2022년 하반기부터 2023년 초까지 반도체 가격 하락 우려로 하락세를 보였으나, 2023년 중반부터는 글로벌 수요 회복 기대감으로 반등하였다. 
- 주요 